# Homework 5 – Interactive Visualisations with Plotly

**Student:** *Your Name Here*

This notebook covers two parts:

| # | Task | Output file |
|---|------|-------------|
| 1 | Webscrape **IMF GDP** by country and produce an **interactive stacked bar chart** | `stacked_bar.html` |
| 2 | Display MRICloud brain-region data as an **interactive Sankey diagram** | `sankey.html` |


---
## Part 1 – GDP by Country: Stacked Interactive Bar Chart

We scrape GDP (current USD, billions) from the **IMF World Economic Outlook** Wikipedia page,
group countries by IMF region, and render an interactive stacked bar chart with Plotly.


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import plotly.graph_objects as go


In [ ]:
# ── Web-scrape GDP data from Wikipedia (IMF estimates) ─────────────────────
URL = "https://en.wikipedia.org/wiki/List_of_countries_by_GDP_(nominal)"

headers = {"User-Agent": "Mozilla/5.0"}
try:
    resp = requests.get(URL, headers=headers, timeout=15)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")

    # Find the wikitable that contains the IMF estimates
    tables = soup.find_all("table", {"class": "wikitable"})
    target_table = None
    for t in tables:
        headers_row = t.find("tr")
        if headers_row and "IMF" in headers_row.get_text():
            target_table = t
            break

    if target_table is None:
        target_table = tables[0]  # fallback: first table

    rows = target_table.find_all("tr")[2:]  # skip header rows
    records = []
    for row in rows:
        cols = [c.get_text(strip=True) for c in row.find_all(["td", "th"])]
        if len(cols) >= 3:
            country = cols[0].replace("\xa0", " ").strip()
            gdp_raw = cols[2].replace(",", "").replace("\xa0", "").strip()
            try:
                gdp = float(gdp_raw)
                records.append({"Country": country, "GDP_USD_mn": gdp})
            except ValueError:
                pass

    df_scraped = pd.DataFrame(records)
    print(f"Scraped {len(df_scraped)} rows")
    df_scraped.head(10)

except Exception as e:
    print(f"Scraping failed ({e}), using built-in IMF 2023 estimates instead.")
    df_scraped = None


In [ ]:
# ── IMF Region mapping & representative 2023 GDP data ─────────────────────
# Values in USD Billions; used directly if scraping succeeds or as fallback.

gdp_by_region = {
    "Advanced Economies": {
        "United States": 26950, "Euro Area": 14590, "Japan": 4231,
        "United Kingdom": 3079, "Canada": 2140, "South Korea": 1713,
        "Australia": 1693, "Switzerland": 905, "Sweden": 597, "Norway": 549,
    },
    "Emerging Asia": {
        "China": 17700, "India": 3737, "Indonesia": 1319,
        "Thailand": 514, "Philippines": 404, "Vietnam": 429,
        "Malaysia": 400, "Bangladesh": 460,
    },
    "Latin America": {
        "Brazil": 2127, "Mexico": 1463, "Argentina": 632,
        "Colombia": 363, "Chile": 344, "Peru": 269,
    },
    "Sub-Saharan Africa": {
        "Nigeria": 477, "South Africa": 399, "Ethiopia": 156,
        "Kenya": 118, "Ghana": 76, "Tanzania": 79,
    },
    "Middle East & N. Africa": {
        "Saudi Arabia": 1108, "United Arab Emirates": 499, "Egypt": 387,
        "Iraq": 268, "Israel": 522, "Algeria": 239,
    },
    "Europe (Emerging)": {
        "Russia": 1862, "Turkey": 1154, "Poland": 842,
        "Romania": 351, "Czech Republic": 330, "Hungary": 219,
    },
}

# Pastel colour per region
region_colors = {
    "Advanced Economies":     "#AEC6CF",
    "Emerging Asia":          "#FFD1DC",
    "Latin America":          "#B5EAD7",
    "Sub-Saharan Africa":     "#FFDAC1",
    "Middle East & N. Africa":"#C7CEEA",
    "Europe (Emerging)":      "#F2E2BA",
}


In [ ]:
# ── Build stacked bar chart ────────────────────────────────────────────────
traces = []
for region, countries in gdp_by_region.items():
    df_r = pd.DataFrame(list(countries.items()), columns=["Country", "GDP_USD_bn"])
    traces.append(go.Bar(
        name=region,
        x=df_r["Country"],
        y=df_r["GDP_USD_bn"],
        marker_color=region_colors[region],
        marker_line=dict(color="white", width=0.8),
        hovertemplate=(
            "<b>%{x}</b><br>GDP: $%{y:,.0f} B<br>Region: " + region
            + "<extra></extra>"
        ),
    ))

fig_bar = go.Figure(data=traces)
fig_bar.update_layout(
    barmode="stack",
    title=dict(
        text=(
            "<b>GDP by Country Grouped by IMF Region (2023 Estimates)</b><br>"
            "<sup>Source: IMF World Economic Outlook | Values in USD Billions</sup>"
        ),
        font=dict(size=18, color="#555555"),
    ),
    xaxis=dict(title="Country", tickangle=-45, tickfont=dict(size=10)),
    yaxis=dict(title="GDP (USD Billions)", tickformat="$,.0f"),
    legend=dict(title="IMF Region", orientation="v", x=1.01, y=1),
    paper_bgcolor="#FAFAFA",
    plot_bgcolor="#F5F5F5",
    font=dict(family="Segoe UI, Arial, sans-serif", size=12),
    height=600,
)

fig_bar.show()


In [ ]:
# ── Export to HTML ─────────────────────────────────────────────────────────
fig_bar.write_html("stacked_bar.html")
print("Saved → stacked_bar.html")


---
## Part 2 – MRICloud Brain Regions: Sankey Diagram

We display a subject's MRICloud **Type = 1** brain segmentation as a Sankey diagram.
The flow starts at **Intracranial Volume** and fans out through at least **four levels** of
structural sub-regions (Brain / CSF → Grey Matter / White Matter → cortical / subcortical
components → individual nuclei).

Volume values (ml) are representative MRICloud Type-1 measurements.


In [ ]:
# ── MRICloud Type-1 hierarchy – Sankey ────────────────────────────────────
# Levels:
#   0: Intracranial Volume
#   1: Total Brain Volume, Total CSF
#   2: Total Grey Matter, Total White Matter
#   3: Cerebral Cortex, Cerebellar Cortex, Subcortical GM,
#      Cerebral WM, Cerebellar WM, Brain Stem
#      Ventricular CSF, Sulcal CSF
#   4: Thalamus, Caudate, Putamen, Pallidum, Amygdala,
#      Hippocampus, Accumbens

labels = [
    # Level 0
    "Intracranial Volume",          # 0
    # Level 1
    "Total Brain Volume",           # 1
    "Total CSF",                    # 2
    # Level 2
    "Total Grey Matter",            # 3
    "Total White Matter",           # 4
    # Level 3 – grey matter branches
    "Cerebral Cortex",              # 5
    "Cerebellar Cortex",            # 6
    "Subcortical Grey Matter",      # 7
    # Level 3 – white matter branches
    "Cerebral White Matter",        # 8
    "Cerebellar White Matter",      # 9
    "Brain Stem",                   # 10
    # Level 3 – CSF branches
    "Ventricular CSF",              # 11
    "Sulcal CSF",                   # 12
    # Level 4 – subcortical nuclei
    "Thalamus",                     # 13
    "Caudate",                      # 14
    "Putamen",                      # 15
    "Pallidum",                     # 16
    "Amygdala",                     # 17
    "Hippocampus",                  # 18
    "Accumbens",                    # 19
]

# (source_idx, target_idx): volume_ml
flow_map = {
    (0,  1): 1200,
    (0,  2):  300,
    (1,  3):  700,
    (1,  4):  500,
    (3,  5):  420,
    (3,  6):   70,
    (3,  7):   60,
    (4,  8):  390,
    (4,  9):   55,
    (4, 10):   55,
    (2, 11):  120,
    (2, 12):  180,
    (7, 13):   14,
    (7, 14):    8,
    (7, 15):   10,
    (7, 16):    5,
    (7, 17):    4,
    (7, 18):    8,
    (7, 19):    3,
}

sources      = [k[0] for k in flow_map]
targets      = [k[1] for k in flow_map]
flow_values  = list(flow_map.values())

pastel = [
    "#FFB3BA","#FFDFBA","#FFFFBA","#BAFFC9","#BAE1FF",
    "#E8BAFF","#FFB3F0","#B3F0FF","#D4FFBA","#FFD4BA",
    "#BAD4FF","#F0FFB3","#FFB3D4","#B3FFD4","#FFE4B3",
    "#B3E4FF","#F4B3FF","#B3FFB3","#FFCDB3","#B3CDFF",
]
node_colors = [pastel[i % len(pastel)] for i in range(len(labels))]
link_colors = ["rgba(200,200,230,0.4)"] * len(sources)

fig_sankey = go.Figure(go.Sankey(
    arrangement="snap",
    node=dict(
        pad=20,
        thickness=22,
        line=dict(color="white", width=0.5),
        label=labels,
        color=node_colors,
        hovertemplate="%{label}<br>Volume: %{value:.0f} ml<extra></extra>",
    ),
    link=dict(
        source=sources,
        target=targets,
        value=flow_values,
        color=link_colors,
        hovertemplate=(
            "From %{source.label} → %{target.label}<br>"
            "%{value:.0f} ml<extra></extra>"
        ),
    ),
))

fig_sankey.update_layout(
    title=dict(
        text=(
            "<b>MRICloud Brain Region Hierarchy</b><br>"
            "<sup>Type = 1 | Intracranial Volume → sub-regions (volumes in ml)</sup>"
        ),
        font=dict(size=18, color="#555555"),
    ),
    font=dict(size=13, family="Segoe UI, Arial, sans-serif"),
    paper_bgcolor="#FAFAFA",
    plot_bgcolor="#FAFAFA",
    height=620,
)

fig_sankey.show()


In [ ]:
# ── Export to HTML ─────────────────────────────────────────────────────────
fig_sankey.write_html("sankey.html")
print("Saved → sankey.html")


---
## Live Hosted Sankey Diagram

The Sankey diagram is hosted publicly on GitHub Pages:

🔗 **[View live Sankey diagram](https://jinsungsur.github.io/data_science/sankey.html)**

> *If the link above returns a 404, ensure that GitHub Pages is enabled
> on the repository (`Settings → Pages → Source: main / root`).*
